Robots with a shape
===================

**Topic**: Path planning for a robot with a shape, which translates and rotates in the plane

Version | Author | Remark
------------ | ------------- | -------------
0.1 | Björn Hein | Example for using a robot with a shape
0.2 | Björn Hein | Reworked: C-obstacles (sampled and exact), reference point, robot with orientation, narrow passage

This worksheet **explains how the planners of the course are applied to a robot with a shape**: first a robot, which only translates (2 DoF), then a robot, which also rotates (3 DoF). For such a robot the C-obstacles can still be computed exactly - a good opportunity to compare them with what the collision checker does.

License is based on Creative Commons: Attribution-NonCommercial 4.0 International (CC BY-NC 4.0) (please check: http://creativecommons.org/licenses/by-nc/4.0/)

**Prerequisite**:

* IP-2-0-Pathplanning-Introduction (configuration space; the disc robot), IP-4-0-Test-Environments (collision checker) and the planners: IP-5 (Basic PRM), IP-6-0 (Visibility-PRM), IP-7-0 (Lazy-PRM)

**Dependencies**:

* IPEnvironmentShapeRobot.py (`ShapeRobot`, `ShapeRobotWithOrientation`, `CollisionCheckerShapeRobot`), IPVISShapeRobot.py (figures of this notebook), IPPerfMonitor.py and the planners IPBasicPRM.py, IPVisibilityPRM.py, IPLazyPRM.py

Important links are:

* General Info: https://www.python.org
* Shapely: https://shapely.readthedocs.io/
* NetworkX: https://networkx.org
* Matplotlib: https://matplotlib.org/stable/

Remember that:

* you have to press ctrl-return or shift-return, to execute the code in the code sections, only then the variables are "generated" and can be used
* you can execute the whole notebook by "Run All"

# Goal

* Understanding the configuration of a robot with a shape: the pose of its **reference point**
* Understanding the collision checker: place the robot, then intersect its geometry with the obstacles
* Understanding the C-obstacles of a translating robot as **Minkowski sum** - and seeing that sampling gives the same result
* Understanding why rotation adds a dimension to the C-space, and how it opens passages
* Experiencing a **narrow passage** in a 3D C-space with different planners

# Idea

A point robot has no extent, a disc robot (IP-2-0) is handled by growing the obstacles by its radius. A robot with an arbitrary shape is described by

* its **geometry**, defined in its own coordinates, and
* its **configuration**: where the origin of these coordinates - the **reference point** - is placed in the work space: $q = (x, y)$, with rotation $q = (x, y, \theta)$.

As for the planar manipulator (IP-10-0) the planners stay unchanged, only the collision checker is exchanged.

In [ ]:
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML
from shapely.geometry import LineString, Polygon, box

from IPEnvironmentShapeRobot import CollisionCheckerShapeRobot, ShapeRobot, ShapeRobotWithOrientation
from IPPerfMonitor import IPPerfMonitor
import IPVISShapeRobot as vis

# A robot which translates (2 DoF)

## Robot and collision checker

The robot is a rectangle of 2.5 x 1. Its reference point is **not** its center: the geometry reaches from -0.5 to 2 in x. `ShapeRobot.setTo(q)` moves the geometry to a configuration.

`CollisionCheckerShapeRobot` (IPEnvironmentShapeRobot.py) is a subclass of `CollisionChecker`; the essential method:

```python
def robotInCollision(self, pos):
    self.robot.setTo(pos)                               # 1. place the robot
    for obstacle in self.scene.values():                # 2. intersect it with every obstacle
        if obstacle.intersects(self.robot.geometry):
            return True
    return False

@IPPerfMonitor
def pointInCollision(self, pos):
    return self.robotInCollision(pos)
```

`lineInCollision` is inherited from `CollisionChecker`: 41 configurations on the line (see "Limits of the model" below).

The scene is the "Trap" of the test suite.

In [ ]:
trapField = dict()
trapField["obs1"] = LineString([(6, 18), (6, 8), (16, 8), (16, 18)]).buffer(1.0)

robotShape = Polygon([(-0.5, -0.5), (2.0, -0.5), (2.0, 0.5), (-0.5, 0.5)])
shapeRobot = ShapeRobot(robotShape, limits=[[0.0, 22.0], [0.0, 22.0]])
environment = CollisionCheckerShapeRobot(trapField, shapeRobot)

configurations = [[3.0, 3.0], [4.0, 12.0], [11.0, 14.0], [17.6, 12.0]]
for q in configurations:
    print(q, "in collision:", environment.pointInCollision(q))

vis.showConfigurations(environment, configurations, "robot at four configurations (white dot: reference point)",
                       labels=["A", "B", "C", "D"])

Configuration B: the reference point is 1.0 away from the wall, but the robot collides. Configuration D: the reference point is only 0.6 away from the wall - no collision, the robot extends to the other side. Whether a configuration collides depends on where the reference point lies in the robot.

## The C-obstacles

Which configurations are in collision? Two ways to find out:

1. **Sampling** (as in IP-2-0 and IP-10-0): the collision check on a grid of configurations.
2. **Exactly**: the robot at $q$ collides with an obstacle $O$, if a point of the robot coincides with a point of the obstacle: $q + r = o$, i.e. $q = o - r$. The C-obstacle is therefore the **Minkowski sum** of the obstacle and the robot mirrored at its reference point:

$$CO = O \oplus (-R) = \{\, o - r \mid o \in O,\; r \in R \,\}$$

For the disc robot this is "growing by the radius" (a disc is symmetric). For a convex robot the Minkowski sum can be computed with shapely (`vis.minkowskiSum`: the robot is swept along the boundary of the obstacle).

In [ ]:
startTime = time.perf_counter()
grid = vis.showCSpace(environment, configurations, resolution=120)
print(f"{grid.size} collision checks, {grid.mean():.0%} of the configurations are in collision")

Gray: the obstacle in the work space, yellow: the C-obstacle. Both methods give the same shape; the sampled one is only as fine as the grid.

* The C-obstacle is **not** the obstacle grown equally in all directions: it is extended by 2 to the left and by 0.5 to the right (the mirrored robot), and by 0.5 up and down.
* The reference point of a colliding configuration (B) lies inside the C-obstacle, the reference point of D outside.
* **The shaped robot in the work space is a point robot in the C-space**: the planners move the reference point between the yellow C-obstacles.

A check, that the collision checker and the Minkowski sum agree, with 3000 random configurations:

In [ ]:
from shapely.geometry import Point

cObstacle = vis.cObstacles(environment)
random.seed(0)
samples = [[random.uniform(0, 22), random.uniform(0, 22)] for i in range(3000)]
different = sum(environment.robotInCollision(q) != cObstacle.intersects(Point(q)) for q in samples)
print("configurations with different results:", different)

## Planning

The planners are used as for the point robot. `runPlanner` fixes the seed and collects some measures with `IPPerfMonitor` (see IP-X-0); the path is checked independently with a fine resolution (`pathInCollision`).

In [ ]:
from IPBasicPRM import BasicPRM
from IPVisibilityPRM import VisPRM
from IPLazyPRM import LazyPRM


def pathInCollision(collisionChecker, path, resolution=0.05):
    "Independent check of a path (list of configurations): True, if a configuration on it is in collision"
    return any(collisionChecker.robotInCollision(q) for q in vis.interpolatePath(path, resolution))


def runPlanner(plannerClass, config, collisionChecker, start, goal, seed):
    "One run of a planner. Returns (planner, solution, measures)"
    random.seed(seed)
    np.random.seed(seed)
    IPPerfMonitor.clearData()
    planner = plannerClass(collisionChecker)
    startTime = time.perf_counter()
    solution = planner.planPath([start], [goal], config) or []
    duration = time.perf_counter() - startTime
    calls = IPPerfMonitor.dataFrame()
    path = vis.solutionConfigurations(planner, solution)
    measures = {"path found": len(solution) > 0,
                "path valid": len(solution) > 0 and not pathInCollision(collisionChecker, path),
                "point checks": int((calls.name == "pointInCollision").sum()),
                "time [s]": round(duration, 2),
                "nodes in path": len(solution)}
    return planner, solution, measures


start, goal = [1.0, 1.0], [10.0, 12.0]

planner, solution, measures = runPlanner(BasicPRM, {"radius": 3, "numNodes": 300}, environment, start, goal, seed=1)
print(measures)
vis.showSolution(environment, planner, solution, "Basic PRM: roadmap of the reference point and C-obstacle")

Left: the roadmap lives in the C-space - its nodes and edges avoid the **yellow** C-obstacle, not only the gray obstacle. Right: the same path in the work space with the robot drawn along it (start green, goal red).

## The reference point matters

The same rectangle, but with the reference point in its center. The robot and the obstacle are the same, the planning problem in the work space is the same - but the C-obstacle is a different one (now symmetric), and a configuration means another pose:

In [ ]:
centeredShape = Polygon([(-1.25, -0.5), (1.25, -0.5), (1.25, 0.5), (-1.25, 0.5)])
centeredEnvironment = CollisionCheckerShapeRobot(trapField, ShapeRobot(centeredShape))

fig, axes = plt.subplots(1, 2, figsize=(10.4, 5.5))
for ax, checker, title in ((axes[0], environment, "reference point at the rear end"),
                           (axes[1], centeredEnvironment, "reference point in the center")):
    vis.drawCObstacles(ax, checker)
    vis.drawObstacles(ax, checker)
    vis.drawRobot(ax, checker, [11.0, 14.0])
    vis.setupAxes(ax, checker, title)
fig.tight_layout()

# A robot which translates and rotates (3 DoF)

`ShapeRobotWithOrientation` additionally rotates the geometry around its reference point. The configuration is $q = (x, y, \theta)$, the C-space is 3-dimensional. The collision checker is the same class: it takes the dimension from the limits of the robot.

## A problem which needs the rotation

A wall with a gap of 2.4. The robot is 2.5 long and 1 wide: lengthwise it fits through the gap, crosswise it does not.

In [ ]:
gapField = dict()
gapField["left"] = box(0.0, 10.5, 9.8, 11.5)
gapField["right"] = box(12.2, 10.5, 22.0, 11.5)

gapStart, gapGoal = [10.0, 4.0], [10.0, 18.0]

translatingEnvironment = CollisionCheckerShapeRobot(gapField, ShapeRobot(robotShape))
vis.showCSpace(translatingEnvironment, [gapStart, gapGoal, [10.0, 11.0]], resolution=120);

For the robot without rotation the C-obstacles of the two walls overlap: the gap is **closed**. Because the C-obstacle is known exactly here, this can even be proven: the free C-space consists of two separated parts.

In [ ]:
freeSpace = box(0, 0, 22, 22).difference(vis.cObstacles(translatingEnvironment))
parts = list(getattr(freeSpace, "geoms", [freeSpace]))
print("connected parts of the free C-space:", len(parts))
print("start and goal in the same part:", any(part.covers(Point(gapStart)) and part.covers(Point(gapGoal)) for part in parts))

planner, solution, measures = runPlanner(BasicPRM, {"radius": 4, "numNodes": 400}, translatingEnvironment, gapStart, gapGoal, seed=1)
print("Basic PRM:", measures)

The planner finds no path - but it cannot tell, whether there is none or whether it only did not find it. (A sampling-based planner is only probabilistically complete.)

## Slices of the 3D C-space

For a fixed orientation $\theta$ the robot only translates: the slice of the C-space at $\theta$ is the Minkowski sum with the **rotated** robot. The C-obstacle changes its shape with $\theta$ - at 0° the gap is closed, towards 90° it opens:

In [ ]:
rotatingRobot = ShapeRobotWithOrientation(robotShape, limits=[[0.0, 22.0], [0.0, 22.0], [-math.pi, math.pi]])
rotatingEnvironment = CollisionCheckerShapeRobot(gapField, rotatingRobot)
print("dimension of the C-space:", rotatingEnvironment.getDim())

vis.showSlices(rotatingEnvironment, [0.0, math.radians(30), math.radians(60), math.radians(75), math.radians(90)],
               position=[11.0, 9.5])

The 3D C-obstacle is the stack of all these slices. A path from the lower to the upper half has to pass the slices near $\pm 90°$: the passage is a small "tunnel" in the 3D C-space - a **narrow passage** (compare the benchmarks "Bottleneck" in IP-4-0, here in 3D and invisible).

## Planning

Three planners, five runs each. Start and goal have the orientation 0°, so the robot has to rotate, pass the gap and rotate back.

In [ ]:
gapStart3, gapGoal3 = gapStart + [0.0], gapGoal + [0.0]

planners = dict()
planners["Basic PRM"] = (BasicPRM, {"radius": 4, "numNodes": 400})
planners["Visibility-PRM"] = (VisPRM, {"ntry": 80})
planners["Lazy-PRM"] = (LazyPRM, {"initialRoadmapSize": 100, "updateRoadmapSize": 50, "kNearest": 8, "maxIterations": 40})

rows = []
for name, (plannerClass, config) in planners.items():
    runs = pd.DataFrame([runPlanner(plannerClass, config, rotatingEnvironment, gapStart3, gapGoal3, seed)[2]
                         for seed in range(5)])
    rows.append({"planner": name, "paths found": int(runs["path found"].sum()),
                 "paths valid": int(runs["path valid"].sum()),
                 "point checks (mean)": int(runs["point checks"].mean()),
                 "time [s] (mean)": round(runs["time [s]"].mean(), 2)})
pd.DataFrame(rows).set_index("planner")

Observations:

* With rotation the problem is solvable - the additional degree of freedom opens the passage.
* The **Lazy-PRM** finds a path in all five runs: it checks only along candidate paths and adds nodes, until a path through the tunnel exists.
* The **Basic PRM** (400 nodes) and the **Visibility-PRM** (80 failed tries) succeed only in some runs: a random configuration has to hit the small tunnel - the right position **and** the right orientation. The Basic PRM spends about 17000 collision checks for the roadmap in the large free rest of the C-space.
* All found paths are valid.

One solution of the Lazy-PRM:

In [ ]:
planner3, solution3, measures3 = runPlanner(*planners["Lazy-PRM"], rotatingEnvironment, gapStart3, gapGoal3, seed=1)
print(measures3)
vis.showSolution(rotatingEnvironment, planner3, solution3, "Lazy-PRM: roadmap projected on x and y", stepSize=0.8)

path3 = np.array(vis.interpolatePath(vis.solutionConfigurations(planner3, solution3), 0.1))
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(path3[:, 1], np.degrees(path3[:, 2]))
ax.axvspan(10.5, 11.5, color="0.85", label="wall")
ax.set_xlabel("y of the reference point")
ax.set_ylabel("orientation [°]")
ax.grid(linewidth=0.3)
ax.legend(frameon=False);

The roadmap is drawn projected on x and y; in this projection its edges cross the wall, because the third coordinate is not visible. The diagram shows the orientation along the path: while the reference point passes the wall (gray), the robot is turned towards 90° (here between about 70° and 85°).

## Animation

(Optional, takes some seconds.)

In [ ]:
HTML(vis.animateSolution(rotatingEnvironment, vis.solutionConfigurations(planner3, solution3)).to_jshtml())

# Limits of the model

* **Mixed units:** the planners measure distances in the C-space with the Euclidean norm of $(x, y, \theta)$ - a mix of length units and radian. The connection radius of the PRM or the nearest neighbour depend on the scale of the scene: 1 rad counts as much as 1 length unit.
* **Orientation is not periodic:** $-\pi$ and $\pi$ are the same orientation, but the planners cannot move "over the border" (as for the joint angles in IP-10-0). A rotation from 170° to -170° takes the long way.
* **Line check:** `lineInCollision` is inherited from `CollisionChecker` and checks 41 configurations per line, however long it is. Position and orientation are interpolated linearly at the same time.
* **Limits of the environment:** the collision checker does not check, whether the robot lies inside the limits. Only the reference point stays inside (the planners sample there) - the robot itself may stick out.
* **Convex robot:** the exact C-obstacles of this notebook (`vis.cObstacles`) need a convex robot. The collision checker works with any shapely geometry.

# Experiments and Exercises

1. Change the shape of the robot (e.g. a triangle or an "L"; an "L" is not convex: compare the sampled C-obstacle with `vis.cObstacles`). How do the C-obstacles change?
2. Reduce the gap to 1.6. How do the slices change? Which planner still finds a path, and how many nodes does it need?
3. Change the limits of the orientation to $[0, \pi]$. Why is the problem still solvable? Does the success rate change?
4. Scale the orientation: plan with the configuration $(x, y, s \cdot \theta)$ for a factor $s$ (subclass of `ShapeRobotWithOrientation`, which divides by $s$ in `setTo`). How does $s$ influence the roadmap and the success of the Basic PRM?
5. Add the limits of the environment to the collision check: the robot must lie completely inside (`shapely.geometry.box` and `covers`).
6. Implement `lineInCollision` with a maximal distance between two checked configurations instead of a fixed number (compare `fk_resolution` in IP-10-0).
7. The robot should move "forwards" like a car: discuss, why this cannot be expressed by C-obstacles (keyword: non-holonomic constraints).